## Deep Learning

### Chapter 9: Customizing Neural Networks Models

Keras provides three main APIs for building neural network models:

*   **Sequential Model**: A simple way to build models layer-by-layer for most problems.
*   **Functional API**: A more flexible way to build models, allowing for complex architectures with multiple inputs/outputs or shared layers.
*   **Model Subclassing**: A highly customizable approach for advanced users who need full control over the model's forward pass.

In [29]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [30]:
# We simulate a simple classification task:
# Each sample has 20 numerical features
# Goal: predict one of 3 classes

num_samples = 1500
num_features = 20
num_classes = 3

# Input data: random Gaussian features
# Think of this as raw "observations"
x = np.random.normal(size=(num_samples, num_features)).astype("float32")

# Hidden "true relationship" (like a real-world unknown function)
true_w = np.random.normal(size=(num_features, num_classes)).astype("float32")

# We generate logits using a linear transformation + noise
# This simulates a realistic imperfect dataset
logits = x @ true_w + 0.25 * np.random.normal(size=(num_samples, num_classes))

# Convert logits → class labels
# This becomes our supervised learning target
y = np.argmax(logits, axis=1).astype("int32")

# Train / validation split
x_train, x_val = x[:1200], x[1200:]
y_train, y_val = y[:1200], y[1200:]

print("x_train.shape:", x_train.shape)
print("y_train.shape:", y_train.shape)
print("x_val.shape:", x_val.shape)
print("y_val.shape:", y_val.shape)

x_train.shape: (1200, 20)
y_train.shape: (1200,)
x_val.shape: (300, 20)
y_val.shape: (300,)


In [31]:
ticket_samples = 1200

# Three independent feature groups
# Each represents a different "view" of a ticket

title_data = np.random.randint(0, 2, size=(ticket_samples, 100)).astype("float32")
body_data = np.random.randint(0, 2, size=(ticket_samples, 1000)).astype("float32")
tags_data = np.random.randint(0, 2, size=(ticket_samples, 12)).astype("float32")

print("title_data.shape:", title_data.shape)
print("body_data.shape:", body_data.shape)
print("tags_data.shape:", tags_data.shape)

title_data.shape: (1200, 100)
body_data.shape: (1200, 1000)
tags_data.shape: (1200, 12)


In [32]:
priority_targets = (0.4 * title_data[:, :10].mean(axis=1)
+ 0.4 * body_data[:, :30].mean(axis=1)
    + 0.2 * tags_data[:, :4].mean(axis=1)
    + 0.05 * np.random.normal(size=(ticket_samples,)))

priority_targets = np.clip(priority_targets, 0, 1).astype("float32").reshape(-1, 1)
print(priority_targets[:10])

[[0.5597859 ]
 [0.4707648 ]
 [0.3956416 ]
 [0.60284203]
 [0.35089976]
 [0.42847198]
 [0.4234427 ]
 [0.36319828]
 [0.42463157]
 [0.5075062 ]]


In [33]:
department_scores = np.stack([title_data[:, :25].mean(axis=1),
                              body_data[:, :250].mean(axis=1),
                              body_data[:, 250:500].mean(axis=1),
                              tags_data[:, :4].mean(axis=1),], axis=1)

# Choose highest scoring department
department_targets = np.argmax(department_scores, axis=1).astype("int32")
print(department_targets[:20])

[3 1 3 3 2 3 1 3 3 3 0 3 0 1 2 0 3 3 0 2]


In [34]:
split = 900

ticket_train_inputs = {"title": title_data[:split],
                       "body": body_data[:split],
                       "tags": tags_data[:split],}

ticket_val_inputs = {"title": title_data[split:],
                     "body": body_data[split:],
                     "tags": tags_data[split:],}

ticket_train_targets = {"priority": priority_targets[:split],
                        "department": department_targets[:split],}

ticket_val_targets = {"priority": priority_targets[split:],
                      "department": department_targets[split:],}

print("Keys of ticket_train_inputs:", ticket_train_inputs.keys())
print("Keys of ticket_train_targets:", ticket_train_targets.keys())
print("Keys of ticket_val_inputs:", ticket_val_inputs.keys())
print("Keys of ticket_val_targets:", ticket_val_targets.keys())

Keys of ticket_train_inputs: dict_keys(['title', 'body', 'tags'])
Keys of ticket_train_targets: dict_keys(['priority', 'department'])
Keys of ticket_val_inputs: dict_keys(['title', 'body', 'tags'])
Keys of ticket_val_targets: dict_keys(['priority', 'department'])


Sequential models are great for simple, linear stacks of layers, while the Functional API is ideal for more complex architectures. Model subclassing is best for researchers and developers who need to implement custom behavior that isn't easily achieved with the other two APIs.

In [35]:
def build_sequential_model():

    # A Sequential model is just a pipeline: Input → Hidden Layers → Output
    inputs = keras.Input(shape=(num_features,), name="features")

    hidden_layer1 = layers.Dense(64, activation="relu")(inputs) # Hidden representation learning
    hidden_layer2 = layers.Dense(32, activation="relu")(hidden_layer1) # Non-linear feature extraction

    # Output layer converts features → probabilities
    outputs = layers.Dense(num_classes, activation="softmax")(hidden_layer2)

    model = keras.Model(inputs=inputs, outputs=outputs, name="sequential_classifier")

    return model

# See model summary
build_sequential_model().summary()

Model: "sequential_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ features (InputLayer)           │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,523 (13.76 KB)

 Trainable params: 3,523 (13.76 KB)

 Non-trainable params: 0 (0.00 B)

In [36]:
seq_model = build_sequential_model()

seq_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", # Sparse categorical crossentropy because labels are integers
                  metrics=["accuracy"])

history_seq = seq_model.fit(x_train, y_train, validation_data=(x_val, y_val),
                            epochs=5, batch_size=32)

Epoch 1/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.4558 - loss: 1.0586 - val_accuracy: 0.6033 - val_loss: 0.9123
Epoch 2/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7167 - loss: 0.7839 - val_accuracy: 0.7967 - val_loss: 0.6741
Epoch 3/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8425 - loss: 0.5493 - val_accuracy: 0.9033 - val_loss: 0.4553
Epoch 4/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8917 - loss: 0.3832 - val_accuracy: 0.9167 - val_loss: 0.3344
Epoch 5/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9242 - loss: 0.2867 - val_accuracy: 0.9233 - val_loss: 0.2657


Keras Functional API allows you to define complex models with non-linear topology, shared layers, and even multiple inputs or outputs. 

It provides a more flexible way to build models compared to the Sequential API, which is limited to linear stacks of layers. With the Functional API, you can create models that have multiple branches, skip connections, and shared layers, making it suitable for a wide range of applications beyond simple feedforward networks.

In [37]:
def build_functional_model():

    # Each input represents a different information source
    title_input = keras.Input(shape=(100,), name="title")
    body_input = keras.Input(shape=(1000,), name="body")
    tags_input = keras.Input(shape=(12,), name="tags")

    '''
    title_input = keras.Input(shape=(100,), name="title")
    -> This creates a symbolic KerasTensor with shape (batch_size, 100). It's a placeholder, not actual data.

    When `title_features = layers.Dense(64, activation="relu", name="title_features")(title_input)` is called:
    Keras uses the symbolic `title_input` (shape: (None, 100)) to infer the input shape for the `Dense` layer.
    The `Dense` layer then automatically creates its weights, for example, a weight matrix of shape (100, 64) for this transformation.

    '''

    # Each branch learns its own representation: (100,) → (64,)
    title_features = layers.Dense(64, activation="relu", name="title_features")(title_input)
    body_features = layers.Dense(64, activation="relu", name="body_features")(body_input)

    # Merge all information sources into one representation
    # Since title_features: (64,); body_features: (64,); tags_input: (12,)
    # Concatenation: 64 + 64 + 12 = 140
    x = layers.Concatenate(name="concat_features")([title_features,
                                                    body_features, tags_input])
    # Print the shape
    print("x.shape:", x.shape)

    # Shared representation layer
    shared = layers.Dense(64, activation="relu", name="shared_features")(x)

    # Output 1: regression (priority score)
    priority_output = layers.Dense(1, activation="sigmoid", name="priority")(shared)

    # Output 2: classification (department)
    department_output = layers.Dense(4, activation="softmax", name="department")(shared)

    return keras.Model(inputs=[title_input, body_input, tags_input],
                       outputs=[priority_output, department_output], name="ticket_model")


# See model summary
build_functional_model().summary()

x.shape: (None, 140)


Model: "ticket_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ title (InputLayer)  │ (None, 100)       │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ body (InputLayer)   │ (None, 1000)      │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ title_features      │ (None, 64)        │      6,464 │ title[0][0]       │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ body_features       │ (None, 64)        │     64,064 │ body[0][0]        │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tags (InputLayer)   │ (None, 12)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_features     │ (None, 140)       │          0 │ title_features[0… │
│ (Concatenate)       │                   │            │ body_features[0]… │
│                     │                   │            │ tags[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_features     │ (None, 64)        │      9,024 │ concat_features[… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ priority (Dense)    │ (None, 1)         │         65 │ shared_features[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ department (Dense)  │ (None, 4)         │        260 │ shared_features[… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 79,877 (312.02 KB)

 Trainable params: 79,877 (312.02 KB)

 Non-trainable params: 0 (0.00 B)

In [38]:
functional_model = build_functional_model()
# functional_model.summary()

functional_model.compile(optimizer="rmsprop",
                         # Each output has its own loss function
                         loss={"priority": "mse", "department":
                               "sparse_categorical_crossentropy" },

                         # Metrics per task
                         metrics={"priority": ["mae"],
                                  "department": ["accuracy"]})

history_ticket = functional_model.fit(ticket_train_inputs,
                                      ticket_train_targets,
                                      validation_data=(ticket_val_inputs,
                                                       ticket_val_targets),
                                      epochs=3, batch_size=32)

x.shape: (None, 140)
Epoch 1/3
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - department_accuracy: 0.3600 - department_loss: 1.3964 - loss: 1.4125 - priority_loss: 0.0150 - priority_mae: 0.0972 - val_department_accuracy: 0.3300 - val_department_loss: 1.3609 - val_loss: 1.3829 - val_priority_loss: 0.0112 - val_priority_mae: 0.0882
Epoch 2/3
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - department_accuracy: 0.4433 - department_loss: 1.2720 - loss: 1.2763 - priority_loss: 0.0137 - priority_mae: 0.0929 - val_department_accuracy: 0.3500 - val_department_loss: 1.4024 - val_loss: 1.3959 - val_priority_loss: 0.0134 - val_priority_mae: 0.0940
Epoch 3/3
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - department_accuracy: 0.5056 - department_loss: 1.1732 - loss: 1.1781 - priority_loss: 0.0135 - priority_mae: 0.0924 - val_department_accuracy: 0.3500 - val_department_loss: 1.3801 - val_loss: 1.3796 - val_priority_loss: 0.0146 - val_priority_mae: 0.0998


Siamese network using the Functional API uses two identical subnetworks to process two different inputs and then combines their outputs to make a final prediction. 

This architecture is particularly useful for tasks like similarity learning, where the model learns to determine how similar two inputs are. 

The subnetworks share weights, which allows the model to learn a common representation for both inputs, making it effective for tasks such as face verification or signature recognition.

In [39]:
def build_siamese_model():

    input_a = keras.Input(shape=(10,), name="item_a")
    input_b = keras.Input(shape=(10,), name="item_b")

    # Shared encoder = same weights applied to both inputs
    shared_encoder = layers.Dense(16, activation="relu", name="shared_encoder")

    encoded_a = shared_encoder(input_a)
    encoded_b = shared_encoder(input_b)

    print("encoded_a.shape:", encoded_a.shape)
    print("encoded_b.shape:", encoded_b.shape)

    # Cosine similarity measures angle between vectors
    similarity = layers.Dot(axes=1, normalize=True, name="cosine_similarity")([encoded_a, encoded_b])

    return keras.Model([input_a, input_b], similarity), shared_encoder


siamese_model, _ = build_siamese_model()
siamese_model.summary()

encoded_a.shape: (None, 16)
encoded_b.shape: (None, 16)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ item_a (InputLayer) │ (None, 10)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_b (InputLayer) │ (None, 10)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_encoder      │ (None, 16)        │        176 │ item_a[0][0],     │
│ (Dense)             │                   │            │ item_b[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cosine_similarity   │ (None, 1)         │          0 │ shared_encoder[0… │
│ (Dot)               │                   │            │ shared_encoder[1… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 176 (704.00 B)

 Trainable params: 176 (704.00 B)

 Non-trainable params: 0 (0.00 B)

Multi-Layer Perceptron (MLP) classifier by subclassing `keras.Model`.  This approach provides the most flexibility, allowing full control over the model's architecture and forward pass (`call` method). 

It includes dense layers and dropout for regularization.

In [40]:
class MLPClassifier(keras.Model):

    def __init__(self, num_classes=3):
        super().__init__(name="subclassed_mlp")

        # Explicit layer definitions
        self.dense1 = layers.Dense(64, activation="relu")
        self.dropout = layers.Dropout(0.3)
        self.dense2 = layers.Dense(32, activation="relu")
        self.out = layers.Dense(num_classes, activation="softmax")

    def call(self, inputs, training=False):
        # Forward pass is fully controlled here
        x = self.dense1(inputs)
        # Dropout behaves differently during training vs inference
        x = self.dropout(x, training=training)
        x = self.dense2(x)

        return self.out(x)

In [41]:
sub_model = MLPClassifier(num_classes=num_classes)

# Build model (required for summary)
_ = sub_model(tf.zeros((1, num_features)))

sub_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

history_sub = sub_model.fit(x_train, y_train,
                            validation_data=(x_val, y_val),
                            epochs=5, batch_size=32)

Epoch 1/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.4142 - loss: 1.0802 - val_accuracy: 0.6733 - val_loss: 0.8989
Epoch 2/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6350 - loss: 0.8608 - val_accuracy: 0.8033 - val_loss: 0.7298
Epoch 3/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7333 - loss: 0.6990 - val_accuracy: 0.8567 - val_loss: 0.5674
Epoch 4/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7808 - loss: 0.5810 - val_accuracy: 0.8767 - val_loss: 0.4407
Epoch 5/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8150 - loss: 0.4746 - val_accuracy: 0.9133 - val_loss: 0.3563
